# Handoff-Style Orchestration: Thesis Support Agents

This notebook demonstrates a lightweight handoff-style orchestration pattern.

Scenario:

A student asks for help with a thesis. A general thesis coach decides which specialist should handle the request next.

Specialists:

- Research Question Agent
- Methodology Agent
- Data Analysis Agent
- Academic Writing Agent
- Citation Agent

Some `picoagents` versions may implement handoff as a dedicated orchestration pattern. This notebook provides a router-based handoff demonstration that is easy to present.

## Setup

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel
from typing import Optional, Literal, List, Dict, Any

from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root or parent directory.
# Adjust this path if your notebook is located elsewhere.
load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    print("API key loaded successfully.")
else:
    print("OPENAI_API_KEY is empty. Deterministic workflow examples can still run, but LLM-agent examples need an API key.")

client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)

API key loaded successfully.


## Create router and specialist agents

In [5]:
class ThesisSupportRequest(BaseModel):
    """Input model for a thesis-support handoff request."""
    user_request: str


def create_handoff_team(model_client: OpenAIChatCompletionClient) -> tuple[Agent, Dict[str, Agent]]:
    """Create the thesis coach router and all specialist agents."""
    thesis_coach = Agent(
        name="thesis_coach",
        instructions=(
            "You are a thesis coach. Decide which specialist should help next. "
            "Choose exactly one label from: research_question, methodology, data_analysis, writing, citation. "
            "Return only the label."
        ),
        model_client=model_client
    )

    research_question_agent = Agent(
        name="research_question_agent",
        instructions="Help refine research problems, research gaps, and research questions.",
        model_client=model_client
    )

    methodology_agent = Agent(
        name="methodology_agent",
        instructions="Help design methodology, data collection, and evaluation strategy.",
        model_client=model_client
    )

    data_analysis_agent = Agent(
        name="data_analysis_agent",
        instructions="Help with data analysis, metrics, tables, and interpretation.",
        model_client=model_client
    )

    academic_writing_agent = Agent(
        name="academic_writing_agent",
        instructions="Help improve academic structure, argumentation, and clarity.",
        model_client=model_client
    )

    citation_agent = Agent(
        name="citation_agent",
        instructions="Help with citations, related work positioning, and reference quality.",
        model_client=model_client
    )

    specialists = {
        "research_question": research_question_agent,
        "methodology": methodology_agent,
        "data_analysis": data_analysis_agent,
        "writing": academic_writing_agent,
        "citation": citation_agent
    }

    return thesis_coach, specialists


thesis_coach, specialists = create_handoff_team(client)

## Define a simple handoff function

### Workflow Diagram

The thesis coach acts as a router and hands off each request to one specialist.

```mermaid
flowchart
    U[Student request] --> C[Thesis Coach router]
    C -->|research_question| RQ[Research Question Agent]
    C -->|methodology| M[Methodology Agent]
    C -->|data_analysis| D[Data Analysis Agent]
    C -->|writing| W[Academic Writing Agent]
    C -->|citation| CI[Citation Agent]
    RQ --> O[Focused specialist response]
    M --> O
    D --> O
    W --> O
    CI --> O
```

How to read it:
- One incoming request goes to the coach first.
- The coach selects exactly one specialist label.
- The selected specialist returns the final focused support.

In [6]:
def select_specialist_key(route_text: str, available_specialists: Dict[str, Agent]) -> str:
    """Resolve router output to a valid specialist key with safe fallback."""
    route = route_text.strip().lower()
    for key in available_specialists:
        if key in route:
            return key
    return "writing"


async def handoff_thesis_support(request: ThesisSupportRequest) -> None:
    """Route a thesis request to one specialist and print the handoff result."""
    print("USER REQUEST:")
    print(request.user_request)
    print("\nTHESIS COACH SELECTS SPECIALIST:")

    route_response = await thesis_coach.run(request.user_request)
    selected_key = select_specialist_key(
        route_response.messages[-1].content,
        specialists,
    )
    print(selected_key)

    selected_agent = specialists[selected_key]

    specialist_prompt = (
        f"The thesis coach handed this request to you.\n"
        f"Original student request: {request.user_request}\n"
        f"Provide focused support in your specialist role."
    )

    print("\nSPECIALIST RESPONSE:")
    response = await selected_agent.run(specialist_prompt)
    print(response.messages[-1].content)

## Run handoff example

In [7]:
request = ThesisSupportRequest(
    user_request="I have a broad topic about AI tutors, but I do not know how to formulate a clear research question."
)

await handoff_thesis_support(request)

USER REQUEST:
I have a broad topic about AI tutors, but I do not know how to formulate a clear research question.

THESIS COACH SELECTS SPECIALIST:
research_question

SPECIALIST RESPONSE:
Good — I can help you turn the broad topic “AI tutors” into a clear, feasible research question. To give the best guidance I need a little information from you; below I also give concrete narrowing strategies, common research gaps you can address, many example research problems and research questions (quantitative, qualitative, mixed-methods, design, and technical), suggestions for methods and measures, and a short checklist for choosing a final question.

First: quick clarifying questions
- What degree level is this for? (Bachelor, Master’s, PhD)
- Which discipline or department (education, HCI, learning sciences, CS, psychology)?
- Which context/population interests you? (K‑12, higher ed, adults, workplace, special needs)
- Do you prefer experimental, design-based, qualitative, computational/ML, or 

## Reflection questions

1. What context should be passed during a handoff?
2. How can handoffs create loops?
3. How can we prevent unproductive handoff chains?

## Student Exercises

Task 1: Add a new specialist ethics_agent and route label ethics.


Task 2: Add loop protection with max_handoffs parameter in handoff_thesis_support.


Task 3: Return a structured dict with selected specialist and response text instead of only printing.


## Solutions

Solution 1

```python
ethics_agent = Agent(
    name="ethics_agent",
    instructions="Help assess ethics, privacy, and responsible research risks.",
    model_client=model_client,
 )

# Add to specialists map
"ethics": ethics_agent
```

Solution 2

```python
async def handoff_thesis_support(request: ThesisSupportRequest, max_handoffs: int = 1) -> None:
    handoff_count = 0
    while handoff_count < max_handoffs:
        route_response = await thesis_coach.run(request.user_request)
        selected_key = select_specialist_key(route_response.messages[-1].content, specialists)
        handoff_count += 1
        break
```

Solution 3

```python
async def handoff_thesis_support(request: ThesisSupportRequest) -> Dict[str, str]:
    route_response = await thesis_coach.run(request.user_request)
    selected_key = select_specialist_key(route_response.messages[-1].content, specialists)
    response = await specialists[selected_key].run(specialist_prompt)
    return {
        "selected_specialist": selected_key,
        "response": response.messages[-1].content,
    }
```